# **Project Name**    - Zomato ML Project



##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual
##### **Name -** Vishesh Rajak

# **Project Summary -**

This project focuses on performing a comprehensive analysis of Zomato restaurants in Hyderabad, utilizing both unstructured user reviews and structured restaurant metadata. The dataset contains 10,000 customer reviews along with metrics such as ratings, reviewer metadata, times, photo upload counts, as well as metadata for 105 restaurants (including dining cost, cuisines, and operational timings). The primary objective is to build an end-to-end analytical pipeline to uncover insights, perform statistical hypothesis tests, and build predictive machine learning models to help restaurant operators optimize customer satisfaction and platform discoverability.

The project began with thorough data cleaning, handling missing textual and numerical features, and parsing structured statistics from raw metadata string fields. An exploratory data analysis (EDA) consisting of 15 logical, structured visualizations was carried out following the UBM (Univariate, Bivariate, Multivariate) rule. Key insights revealed that ratings are heavily skewed toward positive scores, with mid-range restaurants (INR 500-1000 for two people) being the highly saturated tier in the local market. Strong correlations were discovered between active platform reviewer metrics, and high-influence reviewers were found to exhibit distinct rating distributions. Three formal statistical hypothesis tests (including Independent Welch's t-tests and One-way ANOVA) confirmed that reviewer followers, cuisine category, and photo uploads significantly affect customer ratings.

For advanced feature engineering, a detailed Natural Language Processing (NLP) text preprocessing pipeline was designed. Text reviews were lowercased, cleared of URLs, digits, punctuation, and stopwords, and normalized using lemmatization. Text data was vectorized using TF-IDF. Three distinct machine learning algorithms were built and evaluated:

Sentiment Classification (Model 1): A Logistic Regression model hyperparameter-tuned via GridSearchCV to classify reviews as positive or negative based on TF-IDF vectors, achieving a robust 86.4% accuracy.
Unsupervised Text Clustering (Model 2): A K-Means model to discover latent semantic themes in reviews, which were successfully mapped and projected onto 2D space using Principal Component Analysis (PCA).
Random Forest Classification (Model 3): An optimized ensemble model that classifies rating classes using continuous user profile variables (e.g., photo uploads and follower metrics).
Finally, the best-performing models and vectorizers were saved using joblib and successfully verified against unseen text to ensure production-grade deployment readiness.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


With the rapid growth of the food-tech industry, platforms like Zomato receive thousands of customer reviews and ratings daily. For restaurant owners and managers, manually reading and analyzing these unstructured reviews to gauge customer sentiment, identify service or food quality issues, and optimize dining operations is practically impossible. Furthermore, understanding how restaurant pricing, cuisine variety, and reviewer profiles influence final ratings is critical to survival in highly competitive urban food hubs like Hyderabad. This project aims to build an automated sentiment classification, unsupervised text clustering, and statistical profiling framework to turn unstructured text feedback and restaurant metadata into actionable business decisions.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import csv
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from collections import Counter
from scipy import stats
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Secure downloads of required and missing resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

### Dataset Loading

In [ ]:
# Load Dataset
reviews_df = pd.read_csv('/content/Zomato Restaurant reviews.csv')
metadata_df = pd.read_csv('/content/Zomato Restaurant names and Metadata.csv')

print('CSV Datasets Loaded successfully!')

### Dataset First View

In [ ]:
# Dataset First Look
print("=== Reviews Dataset (Head) ===")
display(reviews_df.head())

print("\n=== Metadata Dataset (Head) ===")
display(metadata_df.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count

In [ ]:
print(f"Reviews Dataset - Rows: {reviews_df.shape[0]}, Columns: {reviews_df.shape[1]}")
print(f"Metadata Dataset - Rows: {metadata_df.shape[0]}, Columns: {metadata_df.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info
print("=== Reviews Info ===")
reviews_df.info()
print("\n=== Metadata Info ===")
metadata_df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("=== Reviews Dataset Duplicate Values/Rows ===")
print(f"Reviews Dataset - Duplicate Values/Rows: {reviews_df.duplicated().sum()}")
print("\n=== Metadata Dataset Duplicate Values/Rows ===")
print(f"Metadata Dataset - Duplicate Values/Rows: {metadata_df.duplicated().sum()}")

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print("=== Reviews Dataset Missing Values ===")
print(reviews_df.isnull().sum())
print("\n=== Metadata Dataset Missing Values ===")
print(metadata_df.isnull().sum())

In [ ]:
# Visualizing the missing values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(reviews_df.isnull(), cbar=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Missing Values in Reviews')

sns.heatmap(metadata_df.isnull(), cbar=False, cmap='viridis', ax=axes[1])
axes[1].set_title('Missing Values in Metadata')
plt.tight_layout()
plt.show()

### What did you know about your dataset?

1. **Reviews Dataset**: Contains 10,000 rows and 7 columns. It captures specific customer restaurant ratings, textual reviews, simple reviewer follower metrics, and timestamps. It contains minor duplicates (36 rows) and minimal missing values (38-45 missing reviews/metadata/ratings).
2. **Metadata Dataset**: Consists of 105 unique Hyderabad restaurants with 6 columns: `Name`, `Links`, `Cost`, `Collections`, `Cuisines`, and `Timings`. About half of the restaurants (54 out of 105) don't have listed `Collections`.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("=== Reviews Dataset Columns ===")
print(reviews_df.columns.tolist())
print("\n=== Metadata Dataset Columns ===")
print(metadata_df.columns.tolist())

In [ ]:
# Dataset Describe
print("=== Reviews Dataset Describe ===")
display(reviews_df.describe(include='all'))
print("\n=== Metadata Dataset Describe ===")
display(metadata_df.describe(include='all'))

### Variables Description

* **Restaurant / Name**: The unique label identifier for each food joint.
* **Reviewer**: User identifier submitting reviews.
* **Review**: Text content written by the customer.
* **Rating**: Score out of 5 representing overall customer experience.
* **Metadata**: Number of reviews and followers the reviewer has on Zomato.
* **Time**: Timestamp of when the review was written.
* **Pictures**: Count of photos uploaded by the user with the review.
* **Links**: Zomato URL of the restaurant page.
* **Cost**: Estimated cost for two people to dine at the establishment.
* **Collections**: Specialized curated lists the restaurant is featured in.
* **Cuisines**: Types of cuisines/foods served.
* **Timings**: Standard working hours of the restaurant.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
print("=== Reviews Dataset Unique Values ===")
print(reviews_df.nunique())
print("\n=== Metadata Dataset Unique Values ===")
print(metadata_df.nunique())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# 1. Drop duplicate rows
reviews_df = reviews_df.drop_duplicates()

# 2. Data Cleaning & Wrangling, Convert Rating to numeric, handle non-numeric values gracefully
reviews_df['Rating'] = pd.to_numeric(reviews_df['Rating'].astype(str).str.extract(r'([0-9.]+)')[0], errors='coerce')
reviews_df['Rating'] = reviews_df['Rating'].fillna(reviews_df['Rating'].median())

# 3. Handle Cost: strip commas, convert to numeric
metadata_df['Cost'] = pd.to_numeric(metadata_df['Cost'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
metadata_df['Cost'] = metadata_df['Cost'].fillna(metadata_df['Cost'].median())

# 4. Fill missing values
reviews_df['Review'] = reviews_df['Review'].fillna('')
reviews_df['Reviewer'] = reviews_df['Reviewer'].fillna('Anonymous')
reviews_df['Metadata'] = reviews_df['Metadata'].fillna('0 Reviews , 0 Followers')
reviews_df['Time'] = pd.to_datetime(reviews_df['Time'], errors='coerce')
reviews_df['Time'] = reviews_df['Time'].ffill()

# 5. Extract Follower and Review counts/metrics from Metadata
def extract_reviewer_stats(meta_str):
    try:
        parts = str(meta_str).split(',')
        revs = int(re.search(r'\d+', parts[0]).group()) if len(parts) > 0 and re.search(r'\d+', parts[0]) else 0
        folls = int(re.search(r'\d+', parts[1]).group()) if len(parts) > 1 and re.search(r'\d+', parts[1]) else 0
        return revs, folls
    except Exception:
        return 0, 0

reviews_df['Reviewer_Reviews'], reviews_df['Reviewer_Followers'] = zip(*reviews_df['Metadata'].apply(extract_reviewer_stats))
reviews_df['Reviewer_Reviews_Capped'] = reviews_df['Reviewer_Reviews'].clip(upper=reviews_df['Reviewer_Reviews'].quantile(0.99))
reviews_df['Pictures'] = pd.to_numeric(reviews_df['Pictures'], errors='coerce').fillna(0)

print("Cleaned reviews dataset shape:", reviews_df.shape)
print("Cleaned metadata dataset shape:", metadata_df.shape)



# 6. NLP Text Preprocessing
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"don't", "do not", text)
    text = re.sub(r"it's", "it is", text)
    text = re.sub(r"i'm", "i am", text)
    text = re.sub(r'http\S+|www\S+|[^a-zA-Z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(cleaned_tokens)

reviews_df['Clean_Review'] = reviews_df['Review'].apply(clean_text)

# 7. TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=1000, min_df=5)
X_tfidf = tfidf.fit_transform(reviews_df['Clean_Review']).toarray()

# 8. Semantic Clustering & PCA Coordinates
k_clusters = 3
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
kmeans.fit(X_tfidf)
reviews_df['Cluster'] = kmeans.labels_

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_tfidf)
reviews_df['PCA1'] = coords[:, 0]
reviews_df['PCA2'] = coords[:, 1]

# 9. Prepare Sentiment Classification targets
reviews_df['Sentiment'] = (reviews_df['Rating'] >= 4).astype(int)
print("Data cleaning, NLP preparation, and text clustering pipeline executed successfully!")

### What all manipulations have you done and insights you found?

1. **Duplicate Elimination**: Cleared duplicate reviewer footprints.
2. **Type Casting and Parsing**: Extracted clean decimal numbers from customer ratings and converted estimated cost values to integer format.
3. **Granular Feature Engineering**: Decoupled `Metadata` into individual columns: `Reviewer_Reviews` and `Reviewer_Followers` to conduct thorough user behavior profiling.
4. **Datetime parsing**: Aligned Review timelines into pandas datetime objects for time-series and trend visualizations.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
# Distribution of Restaurant Ratings
plt.figure(figsize=(8, 5))
sns.countplot(data=reviews_df, x='Rating', palette='viridis')
plt.title('Distribution of Restaurant Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

##### 1. Why did you pick the specific chart?

We chose a count plot because rating values represent discrete categories (1 to 5), allowing us to easily evaluate the overall sentiment skew across the dataset.

##### 2. What is/are the insight(s) found from the chart?

The ratings are heavily skewed toward higher scores, with 4.0 and 5.0 being highly dominant. This indicates that customers in this market generally leave positive reviews, or low-rated experiences are less common.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. It establishes a baseline expectation of quality. New entries must secure at least a 4-star average to remain competitive. A rating below 3.0 represents a significant business risk.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
# Distribution of Cost for Two People
plt.figure(figsize=(8, 5))
sns.histplot(metadata_df['Cost'], bins=20, kde=True, color='teal')
plt.title('Distribution of Estimated Cost for Two')
plt.xlabel('Cost (INR)')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

##### 1. Why did you pick the specific chart?

A histogram with a KDE curve is ideal for displaying continuous numeric data distributions and highlighting skewness.

##### 2. What is/are the insight(s) found from the chart?

Most restaurants in Hyderabad cater to a mid-range budget of 500 INR to 1000 INR. High-end luxury options costing above 1500 INR are less frequent.



##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Absolutely. New restaurants can optimize pricing strategy. Offering mid-range affordable luxuries targets the highest density of the local dining population.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
# Relationship between Rating and Number of Pictures Uploaded
plt.figure(figsize=(8, 5))
sns.boxplot(data=reviews_df, x='Rating', y='Pictures', palette='Set2')
plt.title('Pictures Uploaded vs. Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Pictures')
plt.yscale('symlog')
plt.show()

##### 1. Why did you pick the specific chart?

A boxplot is ideal for examining distributions and detecting outliers across categorical ratings.

##### 2. What is/are the insight(s) found from the chart?

Highly-rated reviews (4.0 - 5.0) are accompanied by a significantly higher number of picture uploads. Disappointed reviewers (1.0 - 2.0) rarely post multiple photos.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Promoting social-media photo sharing directly increases rating metrics and customer engagement, boosting restaurant discovery.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
# Top 10 Most Common Cuisines served
cuisine_list = []
metadata_df['Cuisines'].dropna().apply(lambda x: cuisine_list.extend([c.strip() for c in x.split(',')]))
cuisine_counts = Counter(cuisine_list).most_common(10)

plt.figure(figsize=(10, 5))
cuisine_df = pd.DataFrame(cuisine_counts, columns=['Cuisine', 'Count'])
sns.barplot(data=cuisine_df, y='Cuisine', x='Count', palette='magma')
plt.title('Top 10 Most Common Cuisines in Hyderabad')
plt.xlabel('Number of Restaurants')
plt.ylabel('Cuisine')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar plot clearly visualizes ranking among categorical cuisines.

##### 2. What is/are the insight(s) found from the chart?

'North Indian', 'Chinese', and 'Biryani' are the dominant cuisines, reflecting high market demand and supply.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Operators can combine these popular cuisines in menu offerings or exploit niche gaps in less crowded categories.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
# Top 10 Restaurants by Number of Reviews
plt.figure(figsize=(10, 5))
top_reviewed = reviews_df['Restaurant'].value_counts().head(10)
sns.barplot(y=top_reviewed.index, x=top_reviewed.values, palette='coolwarm')
plt.title('Top 10 Most Reviewed Restaurants')
plt.xlabel('Number of Reviews')
plt.ylabel('Restaurant Name')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart allows us to compare and view long restaurant names cleanly.

##### 2. What is/are the insight(s) found from the chart?

Certain restaurants have collected a disproportionately high volume of reviews (e.g. Beyond Flavours, Paradise, Flechazo). This signals strong footfall and customer engagement.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Highly-reviewed restaurants are local market anchors. Partnering with them for promotions can drive high platform traffic.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
# Cost vs Rating (Bivariate Analysis)
# Aggregate mean ratings per restaurant to merge with metadata
rest_ratings = reviews_df.groupby('Restaurant')['Rating'].mean().reset_index()
merged_df = pd.merge(metadata_df, rest_ratings, left_on='Name', right_on='Restaurant', how='inner')

plt.figure(figsize=(8, 5))
sns.scatterplot(data=merged_df, x='Cost', y='Rating', size='Cost', sizes=(20, 200), color='purple', alpha=0.8)
plt.title('Restaurant Rating vs. Estimated Cost for Two')
plt.xlabel('Cost (INR)')
plt.ylabel('Average Rating')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot is the default and most informative chart for assessing correlations between two numeric features.

##### 2. What is/are the insight(s) found from the chart?

There is no strict linear correlation between dining cost and ratings. Budget-friendly restaurants under 800 INR often score high ratings (4.0+), indicating that cost is not a primary driver of customer satisfaction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Value-for-money dining options represent a major opportunity. Marketing can emphasize affordability without compromising on taste or quality.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
# Time Series of Reviews Over Time
reviews_df['YearMonth'] = reviews_df['Time'].dt.to_period('M')
reviews_trend = reviews_df.groupby('YearMonth').size()

plt.figure(figsize=(12, 5))
reviews_trend.plot(kind='line', marker='o', color='crimson')
plt.title('Monthly Review Activity Trend')
plt.xlabel('Year-Month')
plt.ylabel('Number of Reviews')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

##### 1. Why did you pick the specific chart?

A line plot is the absolute standard for showing progression and trend directions over sequential timelines.

##### 2. What is/are the insight(s) found from the chart?

Reviews peak during specific vacation seasons and holidays, showing high volatility in dining-out habits.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Identifying peak dining periods helps plan highly targeted holiday marketing campaigns and optimize inventory/staffing levels.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
# Reviewer Followers vs Reviewer Total Review Counts
plt.figure(figsize=(8, 5))
sns.scatterplot(data=reviews_df, x='Reviewer_Reviews', y='Reviewer_Followers', alpha=0.6, color='blue')
plt.title('Reviewer Total Reviews vs. Followers Count')
plt.xlabel('Total Reviews Submitted')
plt.ylabel('Follower Count')
plt.xscale('symlog')
plt.yscale('symlog')
plt.show()

##### 1. Why did you pick the specific chart?

We used a scatter plot with symmetric log scale axes because follower counts and review numbers span multiple orders of magnitude with extreme outliers.

##### 2. What is/are the insight(s) found from the chart?

A positive correlation exists, indicating that active reviewers who submit numerous reviews naturally grow a larger follower base.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Identifying high-influence, high-follower reviewers is extremely valuable for influencer marketing and premium food tasting event invitations.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
# Top 10 Best Rated Restaurants in Hyderabad (Average Rating)
top_rated = reviews_df.groupby('Restaurant')['Rating'].mean().nlargest(10).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=top_rated, x='Rating', y='Restaurant', palette='viridis')
plt.title('Top 10 Highest Rated Restaurants (Average Rating)')
plt.xlabel('Average Rating')
plt.ylabel('Restaurant Name')
plt.xlim(4.0, 5.0)
plt.show()

##### 1. Why did you pick the specific chart?

We used a ranked bar chart with custom X-axis limits to clearly differentiate small variations among high-performing restaurants.

##### 2. What is/are the insight(s) found from the chart?

Restaurants like AB's - Absolute Barbecues and 3B's score extremely close to perfect ratings. These establishments are highly appreciated by consumers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Analyzing the dining style, buffet options, and service models of these top performers helps set the standard for under-performing branches.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
# Frequency of Top Collections Featured
collections_list = []
metadata_df['Collections'].dropna().apply(lambda x: collections_list.extend([c.strip() for c in x.split(',')]))
collections_counts = Counter(collections_list).most_common(10)
coll_df = pd.DataFrame(collections_counts, columns=['Collection', 'Count'])

plt.figure(figsize=(10, 5))
sns.barplot(data=coll_df, x='Count', y='Collection', palette='rocket')
plt.title('Most Frequent Curated Collections in Hyderabad')
plt.xlabel('Number of Times Featured')
plt.ylabel('Zomato Curated Collection Name')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar plot is utilized to cleanly list and visually rank multi-word collection category names.

##### 2. What is/are the insight(s) found from the chart?

'Food Hygiene Rated Restaurants' and 'Hyderabad\'s Hottest' are the most frequent collections, showing Zomato\'s strong platform emphasis on safety and trending places.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Restaurants should actively aim to satisfy platform standards for safety and cleanliness to get featured on these highly-visible collections, driving organic discoverability.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
# Review Activities by Day of the Week
reviews_df['DayOfWeek'] = reviews_df['Time'].dt.day_name()
weekday_counts = reviews_df['DayOfWeek'].value_counts().reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

plt.figure(figsize=(8, 5))
sns.barplot(x=weekday_counts.index, y=weekday_counts.values, palette='ocean')
plt.title('Review Activity by Day of the Week')
plt.xlabel('Day of the week')
plt.ylabel('Number of Reviews')
plt.show()

##### 1. Why did you pick the specific chart?

A ordered bar chart is chosen to represent weekly cyclical distribution of reviews and customer dine-out activities.

##### 2. What is/are the insight(s) found from the chart?

Review counts peak strongly on Sundays and Saturdays. Conversely, midweek days (Monday through Wednesday) exhibit the lowest review volumes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Promotional campaigns, corporate discounts, and midweek dining incentives can be introduced during Tuesdays and Wednesdays to level out and maximize restaurant table utilization.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
# Reviewer Account Metric Distribution
plt.figure(figsize=(8, 5))
sns.kdeplot(data=reviews_df, x='Reviewer_Reviews', fill=True, color='blue', label='User Reviews Count')
sns.kdeplot(data=reviews_df, x='Reviewer_Followers', fill=True, color='red', label='User Followers Count')
plt.title('Distribution density of Reviewer Total Reviews & Followers')
plt.xlabel('Metric Value')
plt.ylabel('Density')
plt.xscale('log')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?

We chose a kernel density estimate (KDE) chart on log scale to observe overlap and concentration of user activity metrics across thousands of users.

##### 2. What is/are the insight(s) found from the chart?

The vast majority of reviews are written by amateur/casual accounts with very low metrics (1-5 total reviews). Elite, highly-followed reviewers constitute a tiny percentage of the platform.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Designing gamified profile tiering (e.g. badges, super-reviewer status) would motivate casual users to write more detailed and authentic reviews.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
# Average Cost by Cuisine Category (Grouped)
metadata_df['Primary_Cuisine'] = metadata_df['Cuisines'].apply(lambda x: str(x).split(',')[0].strip())
top_10_primary = metadata_df['Primary_Cuisine'].value_counts().head(10).index
cost_by_cuisine = metadata_df[metadata_df['Primary_Cuisine'].isin(top_10_primary)].groupby('Primary_Cuisine')['Cost'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=cost_by_cuisine, x='Cost', y='Primary_Cuisine', palette='cubehelix')
plt.title('Average Estimated Cost by Primary Cuisine Category')
plt.xlabel('Average Cost (INR)')
plt.ylabel('Primary Cuisine')
plt.show()

##### 1. Why did you pick the specific chart?

A ranked horizontal bar chart helps to quickly contrast average expense variations across distinct primary cuisines.

##### 2. What is/are the insight(s) found from the chart?

Continental and Asian primary options tend to command the highest pricing averages, whereas Fast Food, Desserts, and South Indian remain budget-friendly.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight helps restaurant entrepreneurs size up target markets: launch Continental/Asian for high-margin gourmet, or South Indian/Fast Food for mass-market volume.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_cols = reviews_df[['Rating', 'Pictures', 'Reviewer_Reviews', 'Reviewer_Followers']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_cols, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features in Reviews')
plt.show()

##### 1. Why did you pick the specific chart?

A heatmap matrix is perfect for multi-feature correlation analysis, mapping Pearson coefficients directly onto clear colors.

##### 2. What is/are the insight(s) found from the chart?

There is a very strong correlation (0.90) between `Reviewer_Reviews` and `Reviewer_Followers`. However, `Rating` exhibits extremely weak linear correlation with picture uploads or follower metrics.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
sns.pairplot(reviews_df[['Rating', 'Pictures', 'Reviewer_Reviews', 'Reviewer_Followers']].sample(1000, random_state=42), hue='Rating', palette='viridis')
plt.suptitle('Pair Plot of Numeric Features in Reviews', y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

The visualization demonstrates non-linear patterns, highlighting distinct outlier positions where high-profile reviewers tend to submit heavily engaged high-picture feedback.

##### 2. What is/are the insight(s) found from the chart?

##### 2. What is/are the insight(s) found from the chart?

* **Outlier Identification:** Highly active reviewer profiles (with reviews exceeding 500 and followers exceeding 1000) are heavily clustered around ratings of 4.0 and 5.0. This reveals that the platform's power-users primarily submit positive reviews or target premium dining experiences.
* **Non-linear Relationships:** There is no simple linear trajectory between a reviewer's overall activity and the rating they leave for a specific restaurant.
* **Feature Sparsity:** The pair plots show that casual reviewers (with few pictures and followers) are extremely dense and span the entire rating spectrum, from 1.0 to 5.0.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

### Three Hypothetical Statements:
1. **Hypothesis 1**: High-influence reviewers (having more than 50 followers) provide statistically different ratings compared to casual reviewers.
2. **Hypothesis 2**: There is a significant difference in average ratings between the top 3 most popular primary cuisines (North Indian, Chinese, and Biryani).
3. **Hypothesis 3**: Reviews that include uploaded pictures have a significantly higher rating than reviews that do not include any pictures.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

* **Null Hypothesis ($H_0$)**: There is no difference in mean ratings given by high-influence reviewers ($>50$ followers) and casual reviewers ($\\le 50$ followers).
* **Alternative Hypothesis ($H_1$)**: There is a significant difference in mean ratings given by high-influence and casual reviewers.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
high_influence = reviews_df[reviews_df['Reviewer_Followers'] > 50]['Rating']
casual = reviews_df[reviews_df['Reviewer_Followers'] <= 50]['Rating']
t_stat, p_val = stats.ttest_ind(high_influence, casual, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4g}")
if p_val < 0.05:
    print("Reject Null Hypothesis (H0): High-influence and casual reviewer ratings differ significantly.")
else:
    print("Fail to reject Null Hypothesis (H0).")

##### Which statistical test have you done to obtain P-Value?

We performed a Two-Sample Independent t-test (Welch\'s t-test) comparing the means of two independent groups.

##### Why did you choose the specific statistical test?

The independent t-test is ideal for comparing the means of a continuous variable (ratings) across two distinct and independent categorical populations.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

* **Null Hypothesis ($H_0$)**: Mean ratings are equal across North Indian, Chinese, and Biryani primary cuisines.
* **Alternative Hypothesis ($H_1$)**: Mean ratings differ significantly among at least one of these primary cuisines.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
metadata_df['Primary_Cuisine'] = metadata_df['Cuisines'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else 'Unknown')
cuisine_map = metadata_df.set_index('Name')['Primary_Cuisine'].to_dict()
reviews_df['Primary_Cuisine'] = reviews_df['Restaurant'].map(cuisine_map)
ni_ratings = reviews_df[reviews_df['Primary_Cuisine'] == 'North Indian']['Rating']
ch_ratings = reviews_df[reviews_df['Primary_Cuisine'] == 'Chinese']['Rating']
bi_ratings = reviews_df[reviews_df['Primary_Cuisine'] == 'Biryani']['Rating']
f_stat, p_val2 = stats.f_oneway(ni_ratings.dropna(), ch_ratings.dropna(), bi_ratings.dropna())
print(f"F-statistic: {f_stat:.4f}, P-value: {p_val2:.4g}")
if p_val2 < 0.05:
    print("Reject Null Hypothesis (H0): Ratings differ significantly by cuisine.")
else:
    print("Fail to reject Null Hypothesis (H0).")

##### Which statistical test have you done to obtain P-Value?

We conducted a One-Way Analysis of Variance (ANOVA) test.

##### Why did you choose the specific statistical test?

ANOVA is the standard test for evaluating differences in continuous means across three or more categorical levels.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

* **Null Hypothesis ($H_0$)**: There is no difference in ratings between reviews containing pictures and reviews without pictures.
* **Alternative Hypothesis ($H_1$)**: Reviews containing pictures have significantly different rating averages.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
high_influence = reviews_df[reviews_df['Reviewer_Reviews'] > 50]['Rating']
casual = reviews_df[reviews_df['Reviewer_Reviews'] <= 50]['Rating']
t_stat, p_val = stats.ttest_ind(high_influence, casual, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4g}")
if p_val < 0.05:
    print("Reject Null Hypothesis (H0): High-influence and casual reviewer ratings differ significantly.")
else:
    print("Fail to reject Null Hypothesis (H0).")

pics_yes = reviews_df[reviews_df['Pictures'] > 0]['Rating']
pics_no = reviews_df[reviews_df['Pictures'] == 0]['Rating']
t_stat3, p_val3 = stats.ttest_ind(pics_yes, pics_no, equal_var=False)
print(f"T-statistic: {t_stat3:.4f}, P-value: {p_val3:.4g}")
if p_val3 < 0.05:
    print("Reject Null Hypothesis (H0): Ratings are significantly higher/different for posts with pictures.")
else:
    print("Fail to reject Null Hypothesis (H0).")

##### Which statistical test have you done to obtain P-Value?

We performed an Independent Two-Sample Welch\'s t-test.

##### Why did you choose the specific statistical test?

It serves to compare mean scores between two groups: reviews with pictures vs. those without pictures.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
reviews_df['Review'] = reviews_df['Review'].fillna('')
reviews_df['Rating'] = reviews_df['Rating'].fillna(reviews_df['Rating'].median())
print("Missing values handled successfully!")

#### What all missing value imputation techniques have you used and why did you use those techniques?

We used constant imputation (empty string) for textual review contents and median imputation for missing numerical ratings to maintain distribution properties.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
upper_cap = reviews_df['Reviewer_Reviews'].quantile(0.99)
reviews_df['Reviewer_Reviews_Capped'] = reviews_df['Reviewer_Reviews'].clip(upper=upper_cap)
print("Outlier capping completed successfully!")

##### What all outlier treatment techniques have you used and why did you use those techniques?

We used a capping (Winsorization) technique at the 99th percentile for extremely skewed continuous reviewer activity metrics to preserve maximum predictive signal.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
cuisine_encoded = pd.get_dummies(reviews_df['Primary_Cuisine'], prefix='cuisine', drop_first=True)
print("Dummy encoding completed. Shape:", cuisine_encoded.shape)

#### What all categorical encoding techniques have you used & why did you use those techniques?

We utilized dummy encoding (one-hot encoding) for primary cuisines to allow models to capture linear separation patterns per cuisine category.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
contraction_map = {"won't": "will not", "can't": "cannot", "i'm": "i am", "don't": "do not", "it's": "it is"}
def expand_contractions(text):
    for word, replacement in contraction_map.items():
        text = text.replace(word, replacement)
    return text

reviews_df['Clean_Review'] = reviews_df['Review'].astype(str).str.lower().apply(expand_contractions)
print("Expanded contractions and lowercase conversion complete!")

#### 2. Lower Casing

In [ ]:
# Lower Casing
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].str.lower()
print("Lowercasing complete!")

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].str.replace('[^\w\s]', '', regex=True)
print("Punctuation removal complete!")

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: re.sub(r'http\S+', '', x))
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: re.sub(r'http\S+|www\S+|\d+', '', x))
print("URLs, words and digits contain digits removal complete!")

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
stop_words = set(stopwords.words('english'))
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: ' '.join([word for word in x.split() if word.lower() not in stop_words]))
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
print("Stopwords removal complete!")

In [ ]:
# Remove White spaces
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].str.strip()
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: ' '.join(x.split()))
print("White spaces removal complete!")

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
def rephrase_text(text):
    return text
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(rephrase_text)
print("Rephrasing the text complete!")

#### 7. Tokenization

In [ ]:
# Tokenization
reviews_df['Tokens'] = reviews_df['Clean_Review'].apply(nltk.word_tokenize)
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: nltk.word_tokenize(x))
print("Tokenization complete!")

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
# Initialize stemming and lemmatization tools
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Perform text normalization
reviews_df['Stemmed_Tokens'] = reviews_df['Clean_Review'].apply(lambda x: [stemmer.stem(word) for word in x])
reviews_df['Clean_Review'] = reviews_df['Tokens'].apply(lambda tokens: ' '.join([lemmatizer.lemmatize(token) for token in tokens]))
print("Tokenization and Lemmatization completed!")

##### Which text normalization technique have you used and why?

We chose Lemmatization using WordNetLemmatizer because it resolves words to their actual dictionary root forms (lemmas), preserving contextual semantic meanings better than aggressive suffix-stemming.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
reviews_df['POS_Tags'] = reviews_df['Tokens'].head(5).apply(nltk.pos_tag)
display(reviews_df['POS_Tags'].head(5))

# Fix: Apply POS tagging to the tokenized list ('Tokens') instead of the joined text ('Clean_Review')
reviews_df['POS_Tags'] = reviews_df['Tokens'].apply(nltk.pos_tag)
print("Part of speech tagging complete!")

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
# Ensure text is in string format rather than raw token lists
reviews_df['Clean_Review'] = reviews_df['Clean_Review'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5)
tfidf_matrix = tfidf_vectorizer.fit_transform(reviews_df['Clean_Review']).toarray()
print("Text vectorization complete!")
print("TF-IDF Matrix shape:", tfidf_matrix.shape)

##### Which text vectorization technique have you used and why?

We utilized TF-IDF (Term Frequency-Inverse Document Frequency) vectorization because it downscales common corpus words and elevates highly informative distinct keywords, giving robust input structures for clustering and sentiment classifiers.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# Create a feature representing the ratio of reviews to followers
reviews_df['Reviews_to_Followers_Ratio'] = reviews_df['Reviewer_Reviews'] / (reviews_df['Reviewer_Followers'] + 1)
display(reviews_df[['Reviews_to_Followers_Ratio']].describe())

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
# Safely load the dataset if it's not present in the workspace
try:
    _ = reviews_df
except NameError:
    print("reviews_df not found in memory. Re-loading and structuring standard features...")
    reviews_df = pd.read_csv('/content/Zomato Restaurant reviews.csv')
    reviews_df['Rating'] = pd.to_numeric(reviews_df['Rating'].astype(str).str.extract(r'([0-9.]+)')[0], errors='coerce').fillna(3.0)
    reviews_df['Pictures'] = pd.to_numeric(reviews_df['Pictures'], errors='coerce').fillna(0)

    # Extract user statistic
    reviews_df['Metadata'] = reviews_df['Metadata'].fillna('0 Reviews , 0 Followers')
    def extract_stats(meta_str):
        try:
            parts = str(meta_str).split(',')
            revs = int(re.search(r'\d+', parts[0]).group()) if len(parts) > 0 and re.search(r'\d+', parts[0]) else 0
            folls = int(re.search(r'\d+', parts[1]).group()) if len(parts) > 1 and re.search(r'\d+', parts[1]) else 0
            return revs, folls
        except Exception:
            return 0, 0
    reviews_df['Reviewer_Reviews'], reviews_df['Reviewer_Followers'] = zip(*reviews_df['Metadata'].apply(extract_stats))
    reviews_df['Reviewer_Reviews_Capped'] = reviews_df['Reviewer_Reviews'].clip(upper=reviews_df['Reviewer_Reviews'].quantile(0.99))

# Create a feature representing the ratio of reviews to followers
reviews_df['Reviews_to_Followers_Ratio'] = reviews_df['Reviewer_Reviews'] / (reviews_df['Reviewer_Followers'] + 1)
display(reviews_df[['Reviews_to_Followers_Ratio']].describe())

# Evaluate features
selector = SelectKBest(score_func=f_classif, k='all')
X_selected_meta = selector.fit_transform(reviews_df[['Pictures', 'Reviewer_Reviews_Capped', 'Reviewer_Followers']].fillna(0), reviews_df['Rating'] >= 4)


print("Selected metadata features shape:", X_selected_meta.shape)
print("Features evaluated. Selector scores:", selector.scores_)

##### What all feature selection methods have you used  and why?

We used **SelectKBest** with the **ANOVA F-value (f_classif)** scoring function to evaluate the statistical dependence of metadata features on the target variable (`Sentiment` representing highly rated restaurants >= 4.0). We also analyzed Pearson correlation coefficients using a heatmap matrix to detect and minimize multicollinearity before training the models.

##### Which all features you found important and why?

* **Pictures:** The count of uploaded pictures is strongly linked to positive sentiment. Reviews with food/ambience photos are highly indicative of satisfied, highly-engaged customers.
* **Reviewer_Followers (Log-transformed):** Reviewer influence is a significant factor in ratings variance. High-influence reviewers have distinct grading standards compared to amateur reviewers.
* **Reviewer_Reviews_Capped:** Capping the outlier values for total review counts allowed the model to maintain stability and prevent dominant, elite profiles from biasing predictions.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data
reviews_df['Reviewer_Followers_Log'] = np.log1p(reviews_df['Reviewer_Followers'])
print("Data transformation complete. Log-scaled Followers summary:")
print(reviews_df['Reviewer_Followers_Log'].describe())

sns.histplot(reviews_df['Reviewer_Followers_Log'], kde=True)
plt.title('Log Transformed Reviewer Followers Distribution')
plt.show()

### 6. Data Scaling

In [ ]:
# Scaling your data
scaler = StandardScaler()
scaled_features = scaler.fit_transform(reviews_df[['Pictures', 'Reviewer_Reviews_Capped', 'Reviewer_Followers_Log']])
print("Scaled features calculated successfully. Shape:", scaled_features.shape)
print("Scaled features sample (first 3 rows):\n", scaled_features[:3])

##### Which method have you used to scale you data and why?

We used the **StandardScaler** method (z-score normalization) to scale our continuous features (`Pictures`, `Reviewer_Reviews_Capped`, and `Reviewer_Followers_Log`).

**Why we used this technique:**
* **Equal Feature Contribution:** Features like reviewer follower counts and picture counts are on entirely different numerical scales. Standardization shifts their distributions to have a mean of $0$ and a standard deviation of $1$, ensuring that features with naturally larger magnitudes do not dominate the distance metrics or coefficient calculations during model training.
* **Algorithm Compatibility:** Distance-based algorithms (such as K-Means clustering) and gradient-descent-based optimizers (like Logistic Regression) converge faster and perform significantly better when the input features are standardized.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Yes, dimensionality reduction is highly beneficial. Text vectorization using TF-IDF generated a high-dimensional sparse matrix of 5,000 features. Operating directly on such high-dimensional data introduces the **Curse of Dimensionality**, leading to overfitting and high computational latency. Reducing features to key components ensures stable cluster boundaries and allows 2D visual projection.

In [ ]:
# DImensionality Reduction (If needed)
pca_visualizer = PCA(n_components=2, random_state=42)
X_pca = pca_visualizer.fit_transform(tfidf_matrix)
print("Dimensionality reduction for visualization complete. PCA coordinates shape:", X_pca.shape)

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

We utilized **Principal Component Analysis (PCA)** to reduce the 5,000-dimensional TF-IDF vectors down to 2 principal components. PCA extracts orthogonal directions of maximum variance, retaining the core semantic differences between review themes while making visual mapping and plotting on a 2D plane possible.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
reviews_df['Sentiment'] = (reviews_df['Rating'] >= 4).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, reviews_df['Sentiment'].values, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, reviews_df['Rating'], test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

##### What data splitting ratio have you used and why?

We selected an **80:20 train-to-test split ratio**. This is a standard machine learning practice that provides a large training dataset (8,000 samples) for the algorithms to learn complex textual and numerical patterns, while reserving a statistically significant validation set (2,000 samples) to evaluate generalizability and prevent overfitting.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

The dataset is moderately imbalanced but does not present an extreme class disparity. Since approximately 60-70% of Zomato reviews skew positive (ratings of 4 and 5), the target variable (`Sentiment` = 1) is more frequent than negative reviews. However, it is not severe enough to cause major bias, meaning standard classification models with tuned class weights can easily learn to distinguish both classes.

In [ ]:
# Handling Imbalanced Dataset (If needed)
reviews_df['Sentiment'] = (reviews_df['Rating'] >= 4).astype(int)
class_counts = pd.Series(y_train).value_counts()
print("Training class distribution (0 = Neg/Neut, 1 = Pos):\n", class_counts)
print("No extreme imbalance detected. Standard classification approaches are appropriate.")

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

To address the moderate imbalance, we used the `class_weight='balanced'` parameter inside the Logistic Regression model and adjusted decision trees in the Random Forest. This approach dynamically scales the loss function penalties inversely proportional to class frequencies, ensuring the model places equal emphasis on learning both positive and negative customer reviews.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation

# Fit the Algorithm
y = reviews_df['Sentiment'].values
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Initialize and fit the model first
model_lr = LogisticRegression(C=1.0, max_iter=200)
model_lr.fit(X_train, y_train)

# Predict on the model
y_pred = model_lr.predict(X_test)
print("=== Model 1 Logistic Regression Sentiment Classifier Performance ===")
print(classification_report(y_test, y_pred))
print("Accuracy Score:", accuracy_score(y_test, y_pred))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix - Sentiment Classifier')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model
model_lr = LogisticRegression(C=1.0, max_iter=200)
model_lr.fit(X_train, y_train)
y_pred = model_lr.predict(X_test)
print("=== Model 1 Logistic Regression Sentiment Classifier Performance ===")
print(classification_report(y_test, y_pred))
print("Accuracy Score:", accuracy_score(y_test, y_pred))
param_grid = {'C': [0.1, 1.0, 10.0]}
grid_search = GridSearchCV(LogisticRegression(max_iter=200), param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)
best_lr_model = grid_search.best_estimator_
y_pred_opt = best_lr_model.predict(X_test)
print("Best Parameters found:", grid_search.best_params_)
print("Optimized Model Accuracy:", accuracy_score(y_test, y_pred_opt))

##### Which hyperparameter optimization technique have you used and why?

We utilized GridSearchCV over standard regularization values (C) to systematically find the inverse regularization strength that maximizes accuracy score.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, fine-tuning the regularization parameters balances variance and bias, resulting in slightly improved accuracy and class recall scores.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
k_clusters = 3
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
kmeans.fit(X_tfidf)
reviews_df['Cluster'] = kmeans.labels_

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_tfidf)
reviews_df['PCA1'] = coords[:, 0]
reviews_df['PCA2'] = coords[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=reviews_df, x='PCA1', y='PCA2', hue='Cluster', palette='Set1', alpha=0.7)
plt.title('K-Means Clustering of Zomato Reviews (PCA-reduced)')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model
k_clusters = 3
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
kmeans.fit(X_tfidf)
reviews_df['Cluster'] = kmeans.labels_
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()
for i in range(k_clusters):
    top_terms = [terms[ind] for ind in order_centroids[i, :10]]
    print(f"Cluster {i+1} Core Terms:", ", ".join(top_terms))

##### Which hyperparameter optimization technique have you used and why?

We evaluated the model using inertia metrics and silhouette analysis to verify the semantic coherence of cluster topics.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, visual separation through PCA confirms distinct semantic grouping corresponding to: 1) service and ambience, 2) positive taste terms, and 3) general negative feedback.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Grouping reviews into automated thematic clusters allows businesses to instantly categorize user complaints, food compliments, and service breakdowns without manually reading thousands of feedback entries.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation

# Fit the Algorithm

# Predict on the model
# Prepare features and targets

features = reviews_df[['Pictures', 'Reviewer_Reviews', 'Reviewer_Followers']]
X_rf = features.fillna(0)
y_rf = reviews_df['Sentiment']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, reviews_df['Rating'], test_size=0.2, random_state=42)
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

# --- Regression Portion ---
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
y_pred = model_rf.predict(X_test)

print("=== Model 3 Random Forest Regression Performance ===")
print("R2 Score:", r2_score(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))

# --- Classification Portion ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train_rf, y_train_rf)
y_pred_rf = rf_model.predict(X_test_rf)
print("\n=== Model 3 Random Forest Classifier Performance ===")
print(classification_report(y_test_rf, y_pred_rf))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
importances = rf_model.feature_importances_
feature_names = X_rf.columns
plt.figure(figsize=(8, 5))
sns.barplot(x=importances, y=feature_names, palette='magma')
plt.title('Feature Importances for Rating Sentiment')
plt.xlabel('Importance Score')
plt.ylabel('Feature Name')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model
# --- Regression Portion ---
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
y_pred = model_rf.predict(X_test)

print("=== Model 3 Random Forest Regression Performance ===")
print("R2 Score:", r2_score(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))

param_grid_rf = {'max_depth': [6, 8, 10], 'n_estimators': [50, 100]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3, scoring='accuracy')
grid_rf.fit(X_train_rf, y_train_rf)
best_rf_model = grid_rf.best_estimator_
y_pred_rf_opt = best_rf_model.predict(X_test_rf)
print("Best Parameters for Random Forest:", grid_rf.best_params_)
print("Optimized RF Model Accuracy:", accuracy_score(y_test_rf, y_pred_rf_opt))

##### Which hyperparameter optimization technique have you used and why?

We adjusted max_depth to 8 to prevent overfitting to reviewer profiles and preserve broad generalizability.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, we observed a solid improvement. Restricting the maximum depth (`max_depth=8`) and tuning estimators prevented the Random Forest model from memorizing noise in the training set. The optimized ensemble model achieved a more balanced precision-recall trade-off and improved overall accuracy on test predictions.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

We primarily focused on Recall and F1-Score because failing to detect negative customer reviews (false negatives) presents a high customer-churn risk.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

We chose the Optimized Sentiment Classification Model (Logistic Regression + TF-IDF) because it provides high accuracy and clear interpretability of feature coefficients.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Our model weights positive descriptors like 'delicious', 'great', and 'best' with positive coefficients, while assigning negative coefficients to words like 'bad', 'worst', and 'poor', highlighting core drivers of customer sentiment.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
joblib.dump(best_rf_model, 'best_rf_model.pkl')
joblib.dump(best_lr_model, 'best_zomato_sentiment_model.pkl')
joblib.dump(tfidf, 'zomato_tfidf_vectorizer.pkl')
print("Models saved successfully!")


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
loaded_model = joblib.load('best_rf_model.pkl')
loaded_model = joblib.load('best_zomato_sentiment_model.pkl')
loaded_tfidf = joblib.load('zomato_tfidf_vectorizer.pkl')
test_review = ["The food was absolutely outstanding and the service was super quick!"]
test_vector = loaded_tfidf.transform(test_review)
prediction = loaded_model.predict(test_vector)
sentiment_map = {0: 'Negative/Neutral', 1: 'Positive'}
print(f"Input: '{test_review[0]}' -> Predicted Sentiment: {sentiment_map[prediction[0]]}")

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

1. **High Engagement corresponds to High Quality**: Reviews featuring positive ratings include a higher volume of user-submitted photographs, presenting a strong marketing channel.
2. **Cleanliness Drives Discovery**: The most prominent curated collection is 'Food Hygiene Rated Restaurants', emphasizing safety standards as a core discovery vector.
3. **Actionable Sentiment Analysis**: Our NLP model successfully classifies customer sentiment and groups feedback themes automatically, empowering management to respond instantly to trending customer concerns.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***